# Gene-complexity: reproducible project workflow

**Course:** EN.553.740 Machine Learning 1 · **Instructor:** Dr. Sichen Yang  
**Authors:** Jonathan Ma, Sijia Zhu, Jiatong Gao, Yunwei Chai

This is the TA entry point for comparing processed expression, sparse dictionary codes, and frozen scGPT embeddings with logistic classifiers.

**Quick start:** run `python -m pip install -e .`, select the project Python kernel, place the eight input files below, and choose **Run All**. Defaults perform a small real-data training/validation run and generate metrics and figures. They do not tune, freeze models, or evaluate test cells. Supplied embeddings are checked and reused; this notebook does not pretrain scGPT.

**Individual stages:** run setup/configuration first, set `RUN_END_TO_END=False`, then execute the numbered cells as needed. Dependencies are checked. Completed stages are verified and reused; a changed configuration or failed/partial run requires a new `RUN_NAME`. No existing model results are overwritten.

The notebook calls the installed `gene_complexity` Python package directly. Keep the repository and scGPT handoff resources alongside it; command-line subprocesses are not used for model stages.

Download and extract `external.zip` at the repository root and place the matching raw Baron file in `data/processed/baron_raw.h5ad`. The bundle supplies frozen splits, scGPT inputs, and original report snapshot sources. No `code/` or previous `results/` directory is needed. See the root README for the complete minimal layout.


## Required resources

Obtain the compatible raw-data/split and full-embedding bundles from the project authors. The complete bundle does not yet have a documented public URL. See the root README for the public Baron download route and optional scGPT weight/sampling archive link. Do not put `.zip` or weight archives in place of these extracted files:

```text
data/processed/baron_raw.h5ad
external/baron_splits_final/baron_split_manifest.csv
external/baron_splits_final/split_settings.json
external/baron_splits_final/baron_exclusions.csv
external/scgpt/pretraining_handoff/outputs/baron/baron_scgpt_full.npz
external/scgpt/pretraining_handoff/outputs/baron/baron_scgpt_full.metadata.json
external/scgpt/pretraining_handoff/outputs/baron/checkpoint_selection.json
external/scgpt/pretraining_handoff/outputs/baron/preparation_report.json
```

The full workflow needs these files even when only a subset of classifiers is selected. Preserve their bytes and provenance reports. The optional public-data bootstrap below needs R and internet access; the offline route needs neither R nor a GPU.


In [ ]:
from pathlib import Path
import importlib.util
import json
import sys

# Find the repository even when the kernel starts in a subdirectory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").is_file()
             and (p / "gene_complexity/workflow.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook inside the complete gene-complexity repository.")
required = ["numpy", "pandas", "scipy", "sklearn", "torch", "anndata", "matplotlib", "threadpoolctl", "IPython"]
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f"Missing dependencies: {missing}. Run {sys.executable} -m pip install -e {ROOT} and restart the kernel.")
try:
    from gene_complexity import Workflow, api
    from gene_complexity.preprocessing import download_data, preprocess_data
    from gene_complexity._paths import ROOT as PACKAGE_ROOT
except ImportError as exc:
    raise ImportError("Install this checkout with python -m pip install -e . and restart the kernel.") from exc
if PACKAGE_ROOT != ROOT:
    raise ValueError(f"The installed package points to {PACKAGE_ROOT}, not this checkout {ROOT}. Reinstall with pip install -e .")
import pandas as pd
from IPython.display import display, Image, Markdown
print("Repository:", ROOT)
print("Kernel:", sys.executable)


## Configuration

`smoke` uses 256 genes, 8 atoms, and at most 2 dictionary epochs. `standard` uses 2,000 genes, 32 atoms, and at most 10 epochs. Both use the full shared train/validation cohorts and fixed classifier C; neither replaces model selection.

Set optional overrides to integers to replace profile defaults. `ALPHA` is the dictionary L1 penalty; larger values favor smaller L1 codes. `C` is inverse classifier regularization; larger values mean weaker L2 regularization. `THREADS` controls classifier numerical threads; dictionary fitting uses one. `PLOTS` generates exportable PNGs. `METHODS` selects the classifiers.

Use a new `RUN_NAME` for different settings or after a failed run. Reopening an unchanged completed run verifies and reuses artifacts. Set `RUN_END_TO_END=False` to execute individual stage cells instead.


In [ ]:
RUN_NAME = "ta_notebook_smoke_v2"
PROFILE = "smoke"  # "smoke" or "standard"
N_GENES = None     # override the profile's feature count
ATOMS = None       # override dictionary size
EPOCHS = None      # override dictionary epoch ceiling
ALPHA = 1.0
C = 1.0
THREADS = 1
METHODS = ["baseline", "dictionary", "scgpt"]
PLOTS = True
RUN_END_TO_END = True

RAW = ROOT / "data/processed/baron_raw.h5ad"
SPLITS = ROOT / "external/baron_splits_final"
SCGPT = ROOT / "external/scgpt/pretraining_handoff/outputs/baron"
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be a single folder name.")
if PROFILE not in {"smoke", "standard"}:
    raise ValueError("PROFILE must be smoke or standard")
profile_genes, profile_atoms, profile_epochs = (256, 8, 2) if PROFILE == "smoke" else (2000, 32, 10)
OUTPUT = ROOT / "results/workflow" / RUN_NAME


### Optional first-time public-data preparation

Skip this cell's actions when the exact eight-file bundle is supplied. Enable both flags only after confirming the scGPT compatibility/exclusion policy for the raw input. The bootstrap installs missing R packages, downloads/exports counts, and prepares final splits and classical features. Existing features are preserved and may cause the preparation script to reject the run. A newly serialized raw file must match the embedding handoff's provenance; do not edit hashes to force agreement.


In [ ]:
DOWNLOAD_AND_PREPARE = False
SCGPT_COMPATIBILITY_CONFIRMED = False
if DOWNLOAD_AND_PREPARE:
    if not SCGPT_COMPATIBILITY_CONFIRMED:
        raise ValueError("Confirm scGPT compatibility/exclusions before creating final shared splits.")
    if RAW != ROOT / "data/processed/baron_raw.h5ad" or SPLITS != ROOT / "external/baron_splits_final":
        raise ValueError("Bootstrap uses default raw/split paths; use the README CLI for custom destinations.")
    download_data()
    preprocess_data(scgpt_confirmed=True)
else:
    print("Using supplied raw counts and final splits; public-data bootstrap skipped.")


## 1. Validate locations, provenance, and run settings

Missing inputs are listed together. Invalid hashes, incompatible scGPT provenance, an altered existing run, or invalid numerical settings raise errors before model fitting. Detailed array/cohort checks run within the stages. A failed stage keeps its log and partial output; correct the cause and choose a new run name.


In [ ]:
workflow = Workflow(
    output=OUTPUT, raw=RAW, splits=SPLITS, scgpt=SCGPT,
    n_genes=N_GENES if N_GENES is not None else profile_genes,
    atoms=ATOMS if ATOMS is not None else profile_atoms,
    epochs=EPOCHS if EPOCHS is not None else profile_epochs,
    alpha=ALPHA, C=C, threads=THREADS, methods=METHODS, plots=PLOTS,
)
display(pd.Series(workflow.settings, name="Run settings"))
print("Outputs:", OUTPUT)


## 2. End-to-end validation run

This executes preprocessing, scGPT checks, dictionary fitting, selected classifiers, validation metrics, comparison plots, and dictionary interpretation. No final test scores or frozen selections are created. Preprocessing does export transformed test features as part of its normal bundle.


In [ ]:
if RUN_END_TO_END:
    workflow.run_all()
else:
    print("Individual-stage mode: run the cells below in dependency order.")


## 3. Individual stages

These cells also work after the end-to-end run: they verify and reuse completed outputs rather than retraining. To change settings, edit configuration and use a new run name. Run the prerequisite cells first if invoking an individual stage in a new run.

### Classical preprocessing


In [ ]:
workflow.preprocess()
metadata = pd.read_csv(OUTPUT / "features/split_manifest.csv")
display(metadata.head())
display(metadata.groupby(["split", "cell_type"]).size().unstack(fill_value=0))


### Frozen scGPT handoff validation

In [ ]:
if "scgpt" in METHODS:
    workflow.check_scgpt()


### Dictionary fitting and frozen validation encoding

In [ ]:
if "dictionary" in METHODS:
    workflow.fit_dictionary()


### Logistic training

Change the method argument to run just one enabled classifier. A dictionary classifier requires the dictionary stage; scGPT requires its handoff check.


In [ ]:
for method in METHODS:
    workflow.train_classifier(method)


### Validation metrics

In [ ]:
for method in METHODS:
    workflow.evaluate_validation(method)


## 4. Generate and view metrics and figures

Validation metric files, per-class tables, and confusion matrices are saved under `validation/<method>/`. `summary/` contains comparison metrics and optional PNGs. These are this run's outputs, not automatically the settings or scores used in the final report.


In [ ]:
workflow.generate_figures()
display(pd.read_csv(OUTPUT / "summary/summary.csv"))
for method in METHODS:
    display(Markdown(f"### {method}: validation per-class metrics"))
    display(pd.read_csv(OUTPUT / "validation" / method / "per_class.csv", index_col=0))
if PLOTS:
    for path in sorted((OUTPUT / "summary").glob("*.png")):
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path), width=750))


### Gene-weight and cell-type activity figures

The review uses training-only atom selection and displays validation with the same normalization. These descriptive associations are not enrichment results or causal pathways.


In [ ]:
if "dictionary" in METHODS:
    workflow.interpret_dictionary()
    display(pd.read_csv(OUTPUT / "dictionary_review/top_gene_weights.csv").head(20))
    if PLOTS:
        display(Image(filename=str(OUTPUT / "dictionary_review/cell_type_activity.png"), width=1000))


## 5. Optional model selection

Enable to search the configured dictionary grid and logistic C for all three representations. This uses this notebook's preprocessed features: use `PROFILE="standard"` with 2,000 genes for the original classical recipe. A smoke-feature selection run is an execution exercise, not a reproduction of the reported experiment. Review recommendations; selection does not automatically freeze a model.


In [ ]:
RUN_MODEL_SELECTION = False
GRID = ROOT / "gene_complexity/dictionary_learning/configs/dictionary_grid_v2.json"
SELECTION_OUTPUT = OUTPUT / "model_selection"
if RUN_MODEL_SELECTION:
    if not GRID.is_file():
        raise FileNotFoundError(f"Missing dictionary grid: {GRID}")
    workflow.run_step("model_selection", api.tune_models,
        dict(inputs=workflow.config, grid=GRID, output=SELECTION_OUTPUT),
        SELECTION_OUTPUT, requires=["preprocess"])
    display(pd.read_csv(SELECTION_OUTPUT / "recommended_models.csv"))
else:
    print("Model selection skipped.")


## 6. Optional selected-model training, freezing, and final test evaluation

After reviewing model selection, enter the selected settings below and explicitly enable the corresponding actions. `SELECTED_CONFIG` must describe the features/dictionary candidates used for selection. Train each selected model, then set `FREEZE_SELECTION=True` only when choices are final. `RUN_FINAL_TEST=True` requires matching frozen records and never refits. Keep test metrics separate from validation; do not tune on test scores.

The settings are empty by default to prevent accidentally declaring smoke settings final. Each selected method needs C; dictionary also needs a candidate key from the selected config. Example syntax: `{"dictionary": {"C": 1.0, "candidate": "EXACT_CONFIG_KEY"}}` (illustrative, not a recommendation).


In [ ]:
TRAIN_SELECTED = False
FREEZE_SELECTION = False
RUN_FINAL_TEST = False
SELECTED_CONFIG = SELECTION_OUTPUT / "validation_config.json"
SELECTED_SETTINGS = {}  # Populate from reviewed recommendations; no defaults.
FINAL = OUTPUT / "selected"

if TRAIN_SELECTED or FREEZE_SELECTION or RUN_FINAL_TEST:
    if not SELECTED_SETTINGS or set(SELECTED_SETTINGS) - {"baseline", "dictionary", "scgpt"}:
        raise ValueError("Provide reviewed SELECTED_SETTINGS for one or more valid methods.")
    if not SELECTED_CONFIG.is_file():
        raise FileNotFoundError(SELECTED_CONFIG)
    for method, settings in SELECTED_SETTINGS.items():
        model = FINAL / "models" / method
        frozen = FINAL / "validation" / method
        if TRAIN_SELECTED:
            options = dict(config=SELECTED_CONFIG, method=method, C=settings["C"], output=model)
            if method == "dictionary":
                options["candidate"] = settings["candidate"]
            workflow.run_step("selected_train_" + method, api.train_classifier, options, model)
        if FREEZE_SELECTION or RUN_FINAL_TEST:
            training_path = model / "training.json"
            if not training_path.is_file():
                raise FileNotFoundError(f"Train the selected model first: {training_path}")
            trained = json.loads(training_path.read_text())
            if trained["method"] != method or float(trained["C"]) != float(settings["C"]):
                raise ValueError(f"Saved {method} model differs from SELECTED_SETTINGS; train the intended choice in a new run.")
            if method == "dictionary" and trained["candidate"] != settings["candidate"]:
                raise ValueError("Saved dictionary candidate differs from SELECTED_SETTINGS.")
        if FREEZE_SELECTION:
            workflow.run_step("selected_freeze_" + method, api.evaluate_validation,
                dict(model=model, freeze=True, output=frozen), frozen,
                requires=["selected_train_" + method])
        if RUN_FINAL_TEST:
            test_dir = FINAL / "test" / method
            workflow.run_step("selected_test_" + method, api.evaluate_test,
                dict(model=model, selection=frozen / "selection.json", output=test_dir),
                test_dir, requires=["selected_freeze_" + method])
    if RUN_FINAL_TEST:
        workflow.run_step("test_summary", api.summarize,
            dict(evaluations=[FINAL / "test" / m for m in SELECTED_SETTINGS], output=FINAL / "test_summary", plots=PLOTS),
            FINAL / "test_summary", requires=["selected_test_" + m for m in SELECTED_SETTINGS])
        display(pd.read_csv(FINAL / "test_summary/summary.csv"))
        if PLOTS:
            for path in sorted((FINAL / "test_summary").glob("*.png")):
                display(Image(filename=str(path), width=750))
else:
    print("Selected-model training, freezing, and test scoring skipped.")


## 7. Optional donor generalization and initialization analysis

This is a full nested donor study, not part of the default smoke run. It uses raw data, the fixed panel, fold-fitted transforms/scalers, and multiple dictionary initializations. It remains retrospective for the existing cohort. Results include metrics, donor plots, and stability figures; seeds are not independent biological replicates.


In [ ]:
RUN_DONOR_STUDY = False
DONOR_ATOMS = 64
DONOR_ALPHA = 2.0
DONOR_EPOCHS = 10
DONOR_SEEDS = [0, 1, 2]
DONOR = OUTPUT / "donor_study"
if RUN_DONOR_STUDY:
    workflow.run_step("donor_study", api.donor_study,
        dict(raw=RAW, splits=SPLITS, scgpt_dir=SCGPT, atoms=DONOR_ATOMS,
             alpha=DONOR_ALPHA, epochs=DONOR_EPOCHS, seeds=DONOR_SEEDS, output=DONOR), DONOR)
    workflow.run_step("donor_summary", api.summarize_donors,
        dict(study=DONOR, output=OUTPUT / "donor_summary", plots=PLOTS),
        OUTPUT / "donor_summary", requires=["donor_study"])
    display(pd.read_csv(OUTPUT / "donor_summary/summary.csv"))
    if PLOTS:
        for path in sorted((OUTPUT / "donor_summary").glob("*.png")):
            display(Image(filename=str(path), width=900))
else:
    print("Donor study skipped.")


## 8. Optional refresh of the paper's tracked figures and tables

The current run's metrics and exportable figures are already generated in sections 4, 6, and 7. This separate action refreshes **the existing paper's** snapshots from the saved CSV and PNG sources supplied in `external/report_artifacts/`. It does not substitute smoke results into the report.

Enable only when those exact artifacts are present. Missing files cause an error; no model fitting occurs. This intentionally updates tracked `docs/notes/assets/` and its provenance manifest. Inspect the resulting Git diff. The report compiles from tracked snapshots even without local experiment outputs.


In [ ]:
REFRESH_PAPER_ASSETS = False
if REFRESH_PAPER_ASSETS:
    api.build_report_assets()
    for path in sorted((ROOT / "docs/notes/assets").glob("*.png")):
        display(Image(filename=str(path), width=900))
else:
    print("Paper snapshots preserved. This run's figures are in:", OUTPUT)


## Run record and troubleshooting

`notebook_run.json` records settings, software versions, input/source hashes, Python function calls, and output hashes. Stage logs are beside it. A missing or changed dependency raises an exception with the path or prerequisite stage. A failed stage is never silently marked complete or overwritten: fix the error and choose a fresh `RUN_NAME`.

Use a project kernel (`python -m ipykernel install --user --name gene-complexity --display-name "Python (gene-complexity)"`). All Python model stages run in that kernel through package imports; only the optional R exporter launches an external process. Installed packages alone do not supply data or scGPT embeddings. Check the resource tree above if preflight fails.

Keep generated model/data outputs outside Git; clear notebook outputs before committing. The notebook is a reproducible entry point, not an archive of final test results.


In [ ]:
display(pd.DataFrame([
    {"stage": name, "status": entry["status"], "log": entry["log"]}
    for name, entry in workflow.state["stages"].items()
]))
print("Run record:", workflow.state_path)
